In [2]:
import cv2
import numpy as np

class Vision:
    def __init__(self):
        pass

    # Charger une image
    def load(self, path="table2.jpg"):
        img = cv2.imread(path)
        if img is None:
            raise FileNotFoundError(f"Impossible de charger {path}")
        return img

    # Détecter obstacles + robot
    def get_state(self, frame):
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        blur = cv2.GaussianBlur(gray, (5,5), 0)

        # Canny
        edges = cv2.Canny(blur, 30, 120)

        # Contours
        contours, _ = cv2.findContours(
            edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )

        obstacles = []
        robot = None
        max_robot_area = 0

        for c in contours:
            area = cv2.contourArea(c)
            if area < 300:
                continue

            # simplifier en polygone
            eps = 0.02 * cv2.arcLength(c, True)
            poly = cv2.approxPolyDP(c, eps, True)
            pts = [(int(p[0][0]), int(p[0][1])) for p in poly]

            # Robot = petit contour noir (aire moyenne)
            if 300 < area < 5000 and area > max_robot_area:
                max_robot_area = area
                M = cv2.moments(c)
                if M["m00"] != 0:
                    cx = M["m10"] / M["m00"]
                    cy = M["m01"] / M["m00"]
                    robot = (cx, cy)
                continue

            # sinon : obstacle
            obstacles.append(pts)

        return {
            "robot": robot,
            "obstacles": obstacles,
            "rewards": []
        }

    # Dessiner le résultat
    def draw(self, frame, state):
        out = frame.copy()

        # robot
        if state["robot"] is not None:
            x, y = state["robot"]
            cv2.circle(out, (int(x), int(y)), 10, (255, 0, 0), -1)
            cv2.putText(out, "ROBOT", (int(x)+10, int(y)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,0,0), 2)

        # obstacles
        for poly in state["obstacles"]:
            pts = np.array(poly, dtype=np.int32)
            cv2.polylines(out, [pts], True, (0,255,0), 3)

            for (px, py) in poly:
                cv2.circle(out, (px, py), 5, (0,255,0), -1)

        return out


# ---------------------------------------------------------
# UTILISATION
# ---------------------------------------------------------

vision = Vision()

img = vision.load("table2.jpg")
state = vision.get_state(img)
res = vision.draw(img, state)

# ===== FIX MACOS POUR NE PAS FREEZE =====
cv2.imshow("Vision Debug", res)

while True:
    key = cv2.waitKey(1) & 0xFF
    if key == 27 or key == ord('q'):  # ESC ou 'q'
        break

cv2.destroyAllWindows()
cv2.waitKey(1)   # ← évite les fenêtres bloquées sur macOS

-1

Test des algo de vision

